# MLSM Lab 1 — Interactive Training

The notebook handles interaction and presentation. The logic is in `run_lab1.py`, and model definitions are in `model_definitions.py`. Run the cells in order.

In [ ]:
from run_lab1 import (
    load_dataset, select_models, evaluate_models,
    get_confusion_data, create_cv, display_results, plot_confusion_matrix,
    save_run_outputs,
)

## 1. Select a Dataset

In [ ]:
print("1 -> GenreAll | 2 -> features-30-sec | 3 -> features-3-sec")
dataset_choice = input("Dataset [1/2/3]: ").strip()

## 2. Load the Data

In [ ]:
dataset_name, filename, X, y, class_counts = load_dataset(dataset_choice)
print(f"{dataset_name}: {len(y)} samples, {X.shape[1]} features")
display_results(class_counts.to_frame())

## 3. Select a Model

In [ ]:
print("l -> Logistic Regression | k -> k-NN | d -> Decision Tree | a -> ANN | all -> All models")
model_choice = input("Model [l/k/d/a/all]: ").strip()
selected_models = select_models(dataset_name, X.shape[1], model_choice)
print("Models:", ", ".join(selected_models))

## 4. Train and Evaluate

Stratified 10-fold cross-validation is used. This cell may take several minutes.

In [ ]:
cv = create_cv()
results_df = evaluate_models(X, y, selected_models, cv)
print(f"Configurations evaluated: {len(results_df)}")

## 5. Results

In [ ]:
display_results(results_df)

## 6. Select a Row for the Confusion Matrix

The matrix uses *out-of-fold* predictions from cross-validation.

In [ ]:
row_choice = input(f"Row index [0–{len(results_df)-1}] or q to skip: ").strip().lower()

## 7. Confusion Matrix

In [ ]:
fig = None
if row_choice == "q":
    print("Confusion matrix skipped.")
else:
    row_index = int(row_choice)
    matrix, classes, row = get_confusion_data(X, y, selected_models, results_df, row_index, cv)
    print(f"{row['Algorithm']} | CV accuracy: {row['CV accuracy']:.4f} | Overfitting gap: {row['Overfitting gap']:.4f}\nParameters: {row['Parameters']}")
    title = f"{dataset_name} — {row['Algorithm']}"
    fig = plot_confusion_matrix(matrix, classes, title, row["Parameters"])

## 8. Save Results (Optional)

This cell is independent and can be run again after the analysis is complete. If you choose to save, an Excel file containing the results and configuration tables, along with the confusion matrix figure, will be created in `Lab1/results/`.

In [ ]:
save_run_outputs(results_df, dataset_name, selected_models, fig)